# KNN Modeling with Hipparcos Data

The data we are using in this workshop come from the Hipparcos stellar survey. Hipparcos was an ESA-run space mission lasting from 1989 to 1993 that recorded the positions and photometric properties of over one hundred thousand stars within the Milky Way. [You can learn more about this instrument on the ESA website](https://www.cosmos.esa.int/web/hipparcos/tools).

**The goal of this workshop is to develop a machine learning model that can predict the spectral type of a star within the Hipparcos dataset.** The main goal is to explore the uses of KNN modeling, but other methods learned during this summer program may also be explored.


In [1]:
# Setting up the packages used in this workshop
import numpy as np
import matplotlib
import matplotlib.pyplot as plt 
import pandas as pd
import os
cd = os.chdir
pwd = os.getcwd
import warnings
warnings.filterwarnings('ignore')
import time

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

# Open and Examine the Data
The Hipparcos dataset is saved as a CSV file titled `hipparcos-voidmain.csv`. Before working with the data, it's always a good idea to open the file, examine the data, and make some plots to explore what is contained in the set. 

### Objective: 
*Load and check the data.*

In [ ]:
### LOAD THE DATA AS A PANDAS DATAFRAME ### 
# You can check the data using df.info(), display(df), etc.
#
#

## Data Contents
Below is a brief explanation of the headers contained in this dataset. Additional information may be found on the [Hipparcos Input Catalog webpage](https://heasarc.gsfc.nasa.gov/w3browse/all/hic.html).

### Header Info
0. Catalog ( Catalog_Name ) - Catalogue (H=Hipparcos) 
1. HIP ( HIP_Number ) - Identifier (HIP number) 
2. Proxy ( Prox_10asec ) - Proximity flag 
3. RAhms ( RA ) - RA in h m s, ICRS (J1991.25) 
4. DEdms ( Dec ) - Dec in deg ' ", ICRS (J1991.25) 
5. Vmag ( Vmag ) - Magnitude in Johnson V *(Is this apparent or absolute magnitude? How do you know?)*
6. VarFlag ( Var_Flag ) - Coarse variability flag 
7. r_Vmag ( Vmag_Source ) - Source of magnitude 
8. RAdeg ( RA_Deg ) - RA in degrees (ICRS, Epoch-J1991.25) 
9. DEdeg ( Dec_Deg ) - Dec in degrees (ICRS, Epoch-J1991.25) 
10. AstroRef ( Astrom_Ref_Dbl ) - Reference flag for astrometry 
11. Plx ( Parallax ) - Trigonometric parallax (in milli-arcseconds)
12. pmRA ( pm_RA ) - Proper motion in RA 
13. pmDE ( pm_Dec ) - Proper motion in Dec 
14. e_RAdeg ( RA_Error ) - Standard error in RA*cos(Dec_Deg) 
15. e_DEdeg ( Dec_Error ) - Standard error in Dec_Deg 
16. e_Plx ( Parallax_Error ) - Standard error in Parallax 
17. e_pmRA ( pm_RA_Error ) - Standard error in pmRA 
18. e_pmDE ( pm_Dec_Error ) - Standard error in pmDE 
19. DE:RA ( Crl_Dec_RA ) - (DE over RA)xCos(delta) 
20. Plx:RA ( Crl_Plx_RA ) - (Plx over RA)xCos(delta) 
21. Plx:DE ( Crl_Plx_Dec ) - (Plx over DE) 
22. pmRA:RA ( Crl_pmRA_RA ) - (pmRA over RA)xCos(delta) 
23. pmRA:DE ( Crl_pmRA_Dec ) - (pmRA over DE) 
24. pmRA:Plx ( Crl_pmRA_Plx ) - (pmRA over Plx) 
25. pmDE:RA ( Crl_pmDec_RA ) - (pmDE over RA)xCos(delta) 
26. pmDE:DE ( Crl_pmDec_Dec ) - (pmDE over DE) 
27. pmDE:Plx ( Crl_pmDec_Plx ) - (pmDE over Plx) 
28. pmDE:pmRA ( Crl_pmDec_pmRA ) - (pmDE over pmRA) 
29. F1 ( Reject_Percent ) - Percentage of rejected data 
30. F2 ( Quality_Fit ) - Goodness-of-fit parameter 
31. --- ( HIP_Number_repeat ) - HIP number (repetition) 
32. BTmag ( BT_Mag ) - Mean BT magnitude [(Tycho system)](https://ui.adsabs.harvard.edu/scan/manifest/1992A&A...258..211S)
33. e_BTmag ( BT_Mag_Error ) - Standard error on BTmag 
34. VTmag ( VT_Mag ) - Mean VT magnitude [(Tycho system)](https://ui.adsabs.harvard.edu/scan/manifest/1992A&A...258..211S)
35. e_VTmag ( VT_Mag_Error ) - Standard error on VTmag 
36. m_BTmag ( BT_Mag_Ref_Dbl ) - Reference flag for BT and VTmag 
37. B-V ( BV_Color ) - Johnson BV colour 
38. e_B-V ( BV_Color_Error ) - Standard error on BV 
39. r_B-V ( BV_Mag_Source ) - Source of BV from Ground or Tycho 
40. V-I ( VI_Color ) - Colour index in Cousins' system 
41. e_V-I ( VI_Color_Error ) - Standard error on VI 
42. r_V-I ( VI_Color_Source ) - Source of VI 
43. CombMag ( Mag_Ref_Dbl ) - Flag for combined Vmag, BV, VI 
44. Hpmag ( Hip_Mag ) - Median magnitude in Hipparcos system 
45. e_Hpmag ( Hip_Mag_Error ) - Standard error on Hpmag 
46. Hpscat ( Scat_Hip_Mag ) - Scatter of Hpmag 
47. o_Hpmag ( N_Obs_Hip_Mag ) - Number of observations for Hpmag 
48. m_Hpmag ( Hip_Mag_Ref_Dbl ) - Reference flag for Hpmag 
49. Hpmax ( Hip_Mag_Max ) - Hpmag at maximum (5th percentile) 
50. HPmin ( Hip_Mag_Min ) - Hpmag at minimum (95th percentile) 
51. Period ( Var_Period ) - Variability period (days) 
52. HvarType ( Hip_Var_Type ) - Variability type 
53. moreVar ( Var_Data_Annex ) - Additional data about variability 
54. morePhoto ( Var_Curv_Annex ) - Light curve Annex 
55. CCDM ( CCDM_Id ) - CCDM identifier 
56. n_CCDM ( CCDM_History ) - Historical status flag 
57. Nsys ( CCDM_N_Entries ) - Number of entries with same CCDM 
58. Ncomp ( CCDM_N_Comp ) - Number of components in this entry 
59. MultFlag ( Dbl_Mult_Annex ) - Double and or Multiple Systems flag 
60. Source ( Astrom_Mult_Source ) - Astrometric source flag 
61. Qual ( Dbl_Soln_Qual ) - Solution quality flag 
62. m_HIP ( Dbl_Ref_ID ) - Component identifiers 
63. theta ( Dbl_Theta ) - Position angle between components 
64. rho ( Dbl_Rho ) - Angular separation of components 
65. e_rho ( Rho_Error ) - Standard error of rho 
66. dHp ( Diff_Hip_Mag ) - Magnitude difference of components 
67. e_dHp ( dHip_Mag_Error ) - Standard error in dHp 
68. Survey ( Survey_Star ) - Flag indicating a Survey Star 
69. Chart ( ID_Chart ) - Identification Chart 
70. Notes ( Notes ) - Existence of notes 
71. HD ( HD_Id ) - HD number (III 135) 
72. BD ( BD_Id ) - Bonner DM (I 119), (I 122) 
73. CoD ( CoD_Id ) - Cordoba Durchmusterung (DM) (I 114) 
74. CPD ( CPD_Id ) - Cape Photographic DM (I 108) 
75. (V-I)red ( VI_Color_Reduct ) - VI used for reductions 
76. SpType ( Spect_Type ) - Spectral type 
77. r_SpType ( Spect_Type_Source ) - Source of spectral type 

## Data Visualization

Making plots from the data is always a good way to visualize and explore the data. Here, we are going to walk through a couple of ways the data can be visualized. 

Since one of the main purposes of the Hipparcos instrument was to measure the position of stars, one cool way to display the data is to plot the stars on a map of the sky. Fill in and run the code below to see a map of the stars: 

In [ ]:
### INPUT NEEDED ###
# Plotting the positions of stars
ra =  # insert the RA data (in degrees) from your Hipparcose DataFrame 
dec = # insert the Dec data (in degrees) from your Hipparcose DataFrame

plt.figure()
plt.subplot(111, projection="aitoff")
plt.grid(True)
plt.plot(ra, dec, "*", markersize=0.1)

**SPECTRAL TYPES**

The goal of this workshop is to use an ML algorithm to identify the spectral types of stars. In general, stars are often classified into [7 different spectral types](https://astro.unl.edu/naap/hr/hr_background1.html) (O, B, A, F, G, K, and M) based on their brightness (luminosity) and temperature. 
Within the dataset we are working with, you may have noticed that the spectral types are also given a subclass (or subtype) represented as a number after the letter. The subtype gives more information about the relative temperature and type of star (e.g. giant, white dwarf, etc.) the object is. *For simplicity, it is recommended that you use only the letter and not the subclass when classifying each star, although the choice is yours.* 

In astronomy, the 7 main spectral types are represented fairly easily with a Hertzsprung-Russel (HR) Diagram with brightness on the y-axis and temperature on the x-axis. You may see HR diagrams that use different properties on each axis, but these properties are generally just different ways of representing brightness and temperature: 

<img src="https://cdn.britannica.com/17/143617-050-DA3F8537/diagram-Hertzsprung-Russell-Annie-Jump-Cannon-type-order.jpg" width="400"/> <img src="https://i.sstatic.net/6lmV4zBM.jpg" width="400"/>

Notice how these example HR diagrams display absolute magnitude as an alternative to luminosity on the y-axis. Absolute magnitude ($M$) is related to apparent magnitude ($m$) and distance between us and the star in parsecs ($d$) through the equation: 

$$
M = m - 2.5\log\left(\frac{d^{2}}{100}\right) = m - 5\log(d) + 5
$$

*Note: we aren't given distance as one of the data features, but we do have parallax. How do you convert parallax into distances?*


### Objective: 
*Recreate some version of the HR diagram with the Hipparcos data. Make sure the spectral type of each star is represented somehow on the diagram (using only stars that fall within the OBAFGKM classes). Print or plot the number of stars within each spectral category. What biases might you predict could arise in your future model based on these data?* 


In [ ]:
### CREATE AN HR DIAGRAM ###
# You may have to clean your data to get the spectral types of each star
# Plot or print how many of each spectral type exists in this data
# 
#

### (OPTIONAL) Objective: 
*Add color to the map of the stars created above to represent spectral types. Do similar stars form in the same regions, or are they well-dispersed? Will this information come in handy when building your KNN model?*

In [ ]:
### Your code here ###
#
#

# First-pass Model

Once you have a good handle on the data and have cleaned it up a little, you're ready to begin building your KNN model. We will start by using only features as they appear in the original dataset and the OBAFGKM classifications associated with each star as labels. Later, we will explore how feature engineering can improve the results from a simple KNN model.

### Objective: 
*Create a first-pass model using some subset (or all) of the headers in the dataset as features. Be sure to clean the data of NaNs or infinities as needed. Test your model on a test set and determine how well the model performs with `accuracy_score`.*

In [ ]:
### INPUT NEEDED ###
# Set up your features DataFrame or array and the corresponding labels
#
#

# Build the training and test set
= train_test_split( , # feature DataFrame or Array, 
                    , # labels DataFrame or Array
                   test_size = , # fractional size of your test set (or use train_size)
                   random_state= ) # controls random number generator for consistent compilations

# Standardize the feature scaling using z-scores
scaler = StandardScaler()
 = scaler.fit_transform( # Input training features and save to variable
 = scaler.fit_transform( # Input test features and save to variable

# Set up the classifier
knn = KNeighborsClassifier( # Set up parameters for your KNN

# Train the classifier 
knn.fit( # Train the KNN on the rescaled training set

# Test the model with the test set
predict_test = knn.predict( # Test the KNN on the rescaled test set

# Print your accuracy to evaluate your model
acc = accuracy_score( # Input expected vs. predicted labels to calculate the accuracy of the model
print(f"Accuracy: {round(acc*100, 1)}%")

In [ ]:
# If the code above raises AttributeError: 'NoneType' object has no attribute 'split' 
# !pip install threadpoolctl==3.1.0

### Objective: 
*Build a confusion matrix of your results. Where are the biggest losses?*

In [ ]:
### Your code here ###
#
#

### Objective: 
*Plot your results on an HR diagram. How does it compare to the HR diagram created above? Does seeing this representation help identify where the model is struggling?*

In [ ]:
### Your code here ###
#
#

# Improving the Model
Chances are, the first-pass model isn't the most accurate or efficient model you could build for these data. There are a few techniques you can try to improve it. First, models can be optimized such that you are using the best combination of parameters when building the KNN and the most important features during the 'training' step. You can also attempt to engineer new features (as we've done before). 

## Model Optimization 
You can optimize your KNN using [`sklearn.model_selection.GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) to determine which combination of parameters from a provided list results in the best, most efficient model. It works by reading in the type of classifier and a library of parameters and potential values for those parameters. You then fit the grid to your data as you would any other `sklearn` ML model and print off the best-fit parameters.

*NOTE: Given the size of this dataset, `GridSearchCV` may take several minutes to run. It's a good idea to time it, so see if it's worth running again with different grid parameters!*

### (OPTIONAL) Objective: 
*Determine the best parameters to use for your KNN with `GridSearchCV`. To shorten the runtime, feel free to use a subset of the data. Create a new model using these parameters. Does it improve your accuracy and/or runtime?*

In [ ]:
### INPUT NEEDED ###
print("Creating grid...")

grid_results = GridSearchCV(KNeighborsClassifier(), 
                            {'n_neighbors': , # a list of the different numbers of neighbors to test 
                             'algorithm': ['ball_tree', 'kd_tree'], # types of algorithms to test. 'brute' is also an option
                             'leaf_size': , # a list of leaf sized to test
                             'weights': ['uniform', 'distance']} # weighting methods to test
                           )

starttime=time.perf_counter()
print("Fitting grid...")
grid_results.fit( # insert your training data and labels
endtime = time.perf_counter()
print(f"Run time: {round((endtime-starttime)/60, 3)} mins")
print('The best model has {}'.format(grid_results.best_params_))


In [ ]:
### Build a new KNN using the parameters identified above. Does your model improve? ###
#
#

\
\
Another useful tool is [`permutation_importance`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html) from `sklearn.inspection` (which we have used before!). As a reminder, this function determines the average "importance" of each feature by applying random permutations to the features and evaluating how the model prediction changes in response. Higher mean importance values point to the most important features in the model. This can help you reduce your feature list, thereby (potentially) reducing confusion for the model.

It's worth noting that this function is much slower with KNNs than with simpler models, like linear regression, and a dataset this large will likely take **several minutes to run**. It's a good idea to run `permutation_importance` on only a subset of the data. 

### (OPTIONAL) Objective: 
*Determine which features are most important to your KNN model using `permutation_importance`.* 

*NOTE: This function may take a long time to run, so feel free to skip this section and move forward with trial-and-error instead.*

In [ ]:
### INPUT NEEDED ###
print("Calculating importances...")
results = permutation_importance(estimator= , # your trained model
                                 X = , # data on which the importance will be computed
                                 y = , # targets for supervised learning
                                 #scoring = , # name of scoring mechanism to use
                                 n_repeats = , # number of permutations to apply
                                 #n_jobs = , # number of parallel jobs
                                 #sample_weight = # sample weights used in scoring
                                 #max_samples = # number of samples to draw from X in each repeat
                                 random_state= ) # pseudo-random number generator

# Printing an ordered list of features by their importance to the model
# NOTE: depending on what (if any) scoring mechanism chosen above, this code may not work as intended

print("\nFEATURE IMPORTANCE (mean +/- std)\n------------------\n")
for i in results.importances_mean.argsort()[::-1]:
        print(f"{feat_list[i]:<8}"
              f"{results.importances_mean[i]:.3f}"
              f" +/- {results.importances_std[i]:.3f}")


## Feature Engineering

Another method for improving the model is **feature engineering** -- that is, making alterations to the features in the dataset in hopes of creating a more instructive measurement from the data. An example of engineered features are **absolute V-band magnitude** and **distance**, which we derived from parallax and/or apparent V-band magnitude. The feature importances from `permutation_importance` may also give you an idea for what other features can be engineered for an effective model. 

### Objective: 
*Engineer as many new features as you think will benefit your KNN and use them to fit a new model. Feel free to remove any features that aren't serving the model. Are you able to improve your accuracy?*

In [ ]:
### Your code here ###
#
#

### Objective: 
*Plot your model results as an HR diagram. How close to the excepted HR diagram can you get your model? Are there any biases appearing in your results?*

In [ ]:
### Your code here ###
#
#

# Alternative Machine Learning Algorithms

In working through this notebook, you may decide that a KNN is not the best classifier for classifying these stars. If not, feel free to try other machine learning methods that we've learned throughout this summer school on these data. Which one produces the most accurate results?

In [ ]:
### Your code here ###
#
#